### Private Client Services (PCS) Maestro Analysis

#### What We Tried
We initially explored including multiple engagement service codes to expand coverage:
- `11420` - Private Client Services (PCS) - **Required to use Maestro**
- `11421` - Tested but not required to use Maestro (21.8% Maestro coverage vs 82.0% for 11420)
- `10164` - No engagements found in dataset
- `10170` - No engagements found in dataset

#### Final Criteria Used
1. Filter `Engagement Service Code` to include only `11420`
2. Filter `Release Date` after `01/01/2025`
3. Exclude engagements containing: `pof`, `perseus`, `pt`, `ITR`, `BTA`, or `Bison` (case-insensitive)
4. Exclude engagements where `ANSR / Tech Revenue ETD` equals `0`
5. Filter `Engagement Status` to: `Closing`, `Completed`, `Pre-Closing`, `Released`

#### Metrics
Statistics are calculated at the **engagement level** (not client level).

In [7]:
# Consolidate all Maestro engagement data from monthly files
import pandas as pd
import os

directory = './raw/maestro/maestro_data'
all_data = []

for f in sorted(os.listdir(directory)):
    if not f.endswith('.xlsx'):
        continue
    
    path = os.path.join(directory, f)
    df = pd.read_excel(path)
    
    # Handle old format with full data (ClientCode, ClientName, EyEngagementId, EngagementName)
    if 'ClientCode' in df.columns and 'ClientName' in df.columns:
        subset = df[['ClientCode', 'ClientName', 'EyEngagementId', 'EngagementName']].copy()
        subset.columns = ['Client Id', 'Client Name', 'Engagement Id', 'Engagement Name']
        all_data.append(subset)
    # Handle new format with only IDs (Client Id, Engagement Id)
    elif 'Client Id' in df.columns and 'Engagement Id' in df.columns:
        subset = df[['Client Id', 'Engagement Id']].copy()
        subset['Client Name'] = None
        subset['Engagement Name'] = None
        subset = subset[['Client Id', 'Client Name', 'Engagement Id', 'Engagement Name']]
        all_data.append(subset)

# Combine all data
combined = pd.concat(all_data, ignore_index=True)

# Get unique Client Id -> Client Name mappings (from files that have names)
client_map = combined[combined['Client Name'].notna()][['Client Id', 'Client Name']].drop_duplicates()

# Get unique Engagement Id -> Engagement Name mappings
engagement_map = combined[combined['Engagement Name'].notna()][['Engagement Id', 'Engagement Name']].drop_duplicates()

# Get all unique ID combinations and merge in names where available
final = combined[['Client Id', 'Engagement Id']].drop_duplicates()
final = final.merge(client_map, on='Client Id', how='left')
final = final.merge(engagement_map, on='Engagement Id', how='left')

# Reorder and sort
final = final[['Client Id', 'Client Name', 'Engagement Id', 'Engagement Name']]
final = final.sort_values(['Client Id', 'Engagement Id']).reset_index(drop=True)

# Save to Excel
output_path = './raw/maestro/maestro_engagement_data_consolidated.xlsx'
final.to_excel(output_path, index=False)

print(f"Consolidated {len(all_data)} files into {len(final)} unique client/engagement combinations")
print(f"Saved to: {output_path}")

Consolidated 25 files into 10169 unique client/engagement combinations
Saved to: ./raw/maestro/maestro_engagement_data_consolidated.xlsx


In [8]:
import pandas as pd

# Load raw data
raw_df = pd.read_feather('./processed/reduced_coerced_engagement_list.feather')
maestro_debug = pd.read_excel('./raw/maestro/maestro_engagement_data_consolidated.xlsx')
maestro_eng_ids = maestro_debug['Engagement Id'].astype(str).str.strip().drop_duplicates()

# Apply all Private Client Criteria filters to raw_df
print(f'Starting engagements: {len(raw_df)}')

# Rule 1: Filter to Service Code 11420 only
before = len(raw_df)
raw_df = raw_df[raw_df['Engagement Service Code'] == '11420']
print(f'Rule 1 (Service Code 11420): {before - len(raw_df)} removed, {len(raw_df)} remaining')

# Rule 2: Filter Release Date after 01/01/2025
before = len(raw_df)
raw_df = raw_df[raw_df['Release Date'] > '2025-01-01']
print(f'Rule 2 (Release Date > 01/01/2025): {before - len(raw_df)} removed, {len(raw_df)} remaining')

# Rule 3: Exclude engagements containing excluded terms (case-insensitive)
excluded_terms = ['pof', 'perseus', 'pt', 'ITR', 'BTA', 'Bison']
excluded_pattern = '|'.join(excluded_terms)
before = len(raw_df)
raw_df = raw_df[~raw_df['Engagement'].str.contains(excluded_pattern, case=False, na=False)]
print(f'Rule 3 (Exclude {", ".join(excluded_terms)}): {before - len(raw_df)} removed, {len(raw_df)} remaining')

# Rule 4: Remove engagements where ANSR / Tech Revenue ETD equals 0
before = len(raw_df)
raw_df = raw_df[raw_df['ANSR / Tech Revenue ETD'] != 0]
print(f'Rule 4 (ETD != 0): {before - len(raw_df)} removed, {len(raw_df)} remaining')

# Rule 5: Filter Engagement Status to allowed values
before = len(raw_df)
allowed_statuses = ['Closing', 'Completed', 'Pre-Closing', 'Released']
raw_df = raw_df[raw_df['Engagement Status'].isin(allowed_statuses)]
print(f'Rule 5 (Engagement Status filter): {before - len(raw_df)} removed, {len(raw_df)} remaining')

private_filtered = raw_df.copy()

# Get Maestro engagements after applying all filters
maestro_filtered = raw_df[raw_df['Engagement ID'].astype(str).str.strip().isin(maestro_eng_ids)]
print(f"Total engagements after all filters: {len(raw_df)}")
print(f"Maestro engagements after all filters: {len(maestro_filtered)}")

print(f"\nEngagement Status distribution for Maestro engagements:")
print(maestro_filtered['Engagement Status'].value_counts())

Starting engagements: 210873
Rule 1 (Service Code 11420): 172439 removed, 38434 remaining
Rule 2 (Release Date > 01/01/2025): 25892 removed, 12542 remaining
Rule 3 (Exclude pof, perseus, pt, ITR, BTA, Bison): 9400 removed, 3142 remaining
Rule 4 (ETD != 0): 233 removed, 2909 remaining
Rule 5 (Engagement Status filter): 0 removed, 2909 remaining
Total engagements after all filters: 2909
Maestro engagements after all filters: 2424

Engagement Status distribution for Maestro engagements:
Engagement Status
Released       1372
Closing         914
Completed       130
Pre-Closing       8
Name: count, dtype: int64


In [9]:
# DIAGNOSTIC: Check how many engagements exist for each service code and where they drop off
import pandas as pd

raw_df_diag = pd.read_feather('./processed/reduced_coerced_engagement_list.feather')
# Codes we tested (only 11420 is used in final analysis)
tested_codes = ['11420', '11421', '10164', '10170']
excluded_terms = ['pof', 'perseus', 'pt', 'ITR', 'BTA', 'Bison']
excluded_pattern = '|'.join(excluded_terms)

print("=== DIAGNOSTIC: Service Code Analysis (All Tested Codes) ===\n")

# Check raw counts before any filters
print("1. Raw counts by service code (before any filters):")
code_counts = raw_df_diag[raw_df_diag['Engagement Service Code'].isin(tested_codes)].groupby('Engagement Service Code').agg({
    'Engagement ID': 'count',
    'ANSR / Tech Revenue ETD': 'sum'
}).rename(columns={'Engagement ID': 'Count', 'ANSR / Tech Revenue ETD': 'ETD Sum'})
code_counts['ETD Sum'] = code_counts['ETD Sum'].apply(lambda x: f'${x/1e6:,.2f}M')
print(code_counts)
print()

# Apply filters step by step for 11420 only (final criteria)
df = raw_df_diag[raw_df_diag['Engagement Service Code'] == '11420'].copy()

print("=== Filter Pipeline for 11420 Only ===\n")
print(f"Starting count: {len(df)}")

print("2. After Release Date filter (> 01/01/2025):")
df_date = df[df['Release Date'] > '2025-01-01']
print(f"   {len(df_date)} remaining")

print(f"3. After exclusion filter ({', '.join(excluded_terms)}):")
df_excl = df_date[~df_date['Engagement'].str.contains(excluded_pattern, case=False, na=False)]
print(f"   {len(df_excl)} remaining")

print("4. After ETD != 0 filter:")
df_etd = df_excl[df_excl['ANSR / Tech Revenue ETD'] != 0]
print(f"   {len(df_etd)} remaining")

print("5. After Status filter:")
allowed_statuses = ['Closing', 'Completed', 'Pre-Closing', 'Released']
df_status = df_etd[df_etd['Engagement Status'].isin(allowed_statuses)]
print(f"   {len(df_status)} remaining")

print(f"\n6. Final ETD for 11420: ${df_status['ANSR / Tech Revenue ETD'].sum()/1e6:,.2f}M")

=== DIAGNOSTIC: Service Code Analysis (All Tested Codes) ===

1. Raw counts by service code (before any filters):
                         Count   ETD Sum
Engagement Service Code                 
11420                    38434  $780.75M
11421                     3379  $379.06M

=== Filter Pipeline for 11420 Only ===

Starting count: 38434
2. After Release Date filter (> 01/01/2025):
   12542 remaining
3. After exclusion filter (pof, perseus, pt, ITR, BTA, Bison):
   3142 remaining
4. After ETD != 0 filter:
   2909 remaining
5. After Status filter:
   2909 remaining

6. Final ETD for 11420: $162.05M


In [10]:
# DIAGNOSTIC: Check Maestro coverage for 11420
maestro_df_diag = pd.read_excel('./raw/maestro/maestro_consolidated.xlsx')
maestro_ids_diag = maestro_df_diag['Engagement Id'].astype(str).str.strip().drop_duplicates()

df_status['In Maestro'] = df_status['Engagement ID'].astype(str).str.strip().isin(maestro_ids_diag)

print("=== DIAGNOSTIC: Maestro Coverage for 11420 (Release Date > 01/01/2025) ===\n")
maestro_summary = df_status.groupby('In Maestro').agg({
    'Engagement ID': 'count',
    'ANSR / Tech Revenue ETD': 'sum'
}).rename(columns={'Engagement ID': 'Count', 'ANSR / Tech Revenue ETD': 'ETD'})
maestro_summary['ETD'] = maestro_summary['ETD'].apply(lambda x: f'${x/1e6:,.2f}M')
print(maestro_summary)
print()

in_maestro = df_status['In Maestro'].sum()
total = len(df_status)
pct = (in_maestro / total) * 100
print(f"Maestro coverage rate for 11420: {in_maestro}/{total} ({pct:.1f}%)")

=== DIAGNOSTIC: Maestro Coverage for 11420 (Release Date > 01/01/2025) ===

            Count       ETD
In Maestro                 
False         485   $31.87M
True         2424  $130.18M

Maestro coverage rate for 11420: 2424/2909 (83.3%)


In [11]:
# Read Maestro engagement IDs and mark presence using left-hand data only
maestro_df = pd.read_excel('./raw/maestro/maestro_consolidated.xlsx')
# Keep only the key and deduplicate to avoid creating multiple matches per left row
maestro_df = maestro_df[['Engagement Id']].drop_duplicates()

# Avoid a full merge that can duplicate left rows when the right-side has multiple matches.
# Instead, copy the left dataframe and add a boolean flag indicating membership in Maestro.
pcs_merged = private_filtered
# Ensure types align (both strings) and strip whitespace to avoid false negatives
pcs_merged['Engagement ID'] = pcs_merged['Engagement ID'].astype(str).str.strip()
maestro_ids = maestro_df['Engagement Id'].astype(str).str.strip()
pcs_merged['In Maestro'] = pcs_merged['Engagement ID'].isin(maestro_ids)

# Calculate ETD totals (by engagement)
pcs_total_etd = pcs_merged['ANSR / Tech Revenue ETD'].sum()
pcs_total_count = pcs_merged['ANSR / Tech Revenue ETD'].count()
pcs_maestro_etd = pcs_merged.loc[pcs_merged['In Maestro'], 'ANSR / Tech Revenue ETD'].sum()
pcs_maestro_count = pcs_merged.loc[pcs_merged['In Maestro'], 'ANSR / Tech Revenue ETD'].count()

print(f"\nTotal PCS ANSR / Tech Revenue ETD: ${pcs_total_etd:,.2f}")
print(f"Maestro PCS ANSR / Tech Revenue ETD: ${pcs_maestro_etd:,.2f}")
print(f"Total PCS Engagements: {pcs_total_count}")
print(f"Maestro Engagements: {pcs_maestro_count}")

# Calculate key metrics (by engagement)
maestro_engagement_pct = (pcs_maestro_count / pcs_total_count) * 100
maestro_revenue_pct = (pcs_maestro_etd / pcs_total_etd) * 100
maestro_revenue_millions = pcs_maestro_etd / 1_000_000
pcs_total_revenue_millions = pcs_total_etd / 1_000_000

print(f"\n--- Key Metrics (by Engagement) ---")
print(f"% of Engagements in Maestro: {maestro_engagement_pct:.1f}%")
print(f"% of Revenue Supported by Maestro: {maestro_revenue_pct:.1f}%")
print(f"Total PCS Revenue: ${pcs_total_revenue_millions:,.2f}M")
print(f"Maestro Supported Revenue: ${maestro_revenue_millions:,.2f}M")

# Save the left-only merged result (keeps only rows from the left dataframe)
# add a timestamp to the filename to avoid overwriting previous runs
timestamp = pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')
# pcs_merged.to_excel(f'./review/maestro/pcs_merged_engagements_{timestamp}.xlsx', index=False)


Total PCS ANSR / Tech Revenue ETD: $162,052,975.48
Maestro PCS ANSR / Tech Revenue ETD: $130,179,357.51
Total PCS Engagements: 2909
Maestro Engagements: 2424

--- Key Metrics (by Engagement) ---
% of Engagements in Maestro: 83.3%
% of Revenue Supported by Maestro: 80.3%
Total PCS Revenue: $162.05M
Maestro Supported Revenue: $130.18M


In [12]:
# Export raw data with Key Metrics summary sheet (using Excel formulas)
from openpyxl.worksheet.table import Table, TableStyleInfo
from openpyxl.utils import get_column_letter
from openpyxl.styles import Font, Alignment, Border, Side, PatternFill

# Select columns to export
export_df = pcs_merged[['Client ID', 'Client', 'Account Channel', 'Engagement ID', 
                        'Engagement Status', 'Engagement Service Code', 'Release Date',
                        'Engagement', 'ANSR / Tech Revenue ETD', 'ANSR / Tech Revenue FYTD',
                        'TER ETD', 'TER FYTD', 'In Maestro']].copy()

output_path = f'./review/maestro/Maestro_Adoption_{timestamp}.xlsx'

with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    # Write raw data first
    export_df.to_excel(writer, index=False, sheet_name='raw')
    
    # Add Excel table formatting to raw sheet
    ws_raw = writer.sheets['raw']
    table_range = f'A1:{get_column_letter(len(export_df.columns))}{len(export_df) + 1}'
    table = Table(displayName='PCSMergedEngagements', ref=table_range)
    style = TableStyleInfo(name='TableStyleMedium2', showFirstColumn=False,
                           showLastColumn=False, showRowStripes=True, showColumnStripes=False)
    table.tableStyleInfo = style
    ws_raw.add_table(table)
    
    # Create Key Metrics sheet with Excel formulas
    wb = writer.book
    ws_metrics = wb.create_sheet('Key Metrics', 0)  # Insert at position 0 (first sheet)
    
    # Define styles
    header_fill = PatternFill(start_color='4472C4', end_color='4472C4', fill_type='solid')
    header_font = Font(bold=True, color='FFFFFF')
    metric_font = Font(bold=True)
    
    # Set column widths
    ws_metrics.column_dimensions['A'].width = 35
    ws_metrics.column_dimensions['B'].width = 25
    
    # Header row
    ws_metrics['A1'] = 'Metric'
    ws_metrics['B1'] = 'Value'
    ws_metrics['A1'].fill = header_fill
    ws_metrics['B1'].fill = header_fill
    ws_metrics['A1'].font = header_font
    ws_metrics['B1'].font = header_font
    
    # Data row count (for formula references)
    data_rows = len(export_df) + 1  # +1 for header
    etd_col = 'I'  # ANSR / Tech Revenue ETD column
    maestro_col = 'M'  # In Maestro column
    
    # Metric rows with Excel formulas
    metrics = [
        ('Total Engagements', f'=COUNTA(raw!A2:A{data_rows})', None),
        ('Maestro Engagements', f'=COUNTIF(raw!{maestro_col}2:{maestro_col}{data_rows},TRUE)', None),
        ('% Engagements in Maestro', f'=B3/B2', '0.0%'),
        ('', '', None),  # Spacer row
        ('Total PCS Revenue (ETD)', f'=SUM(raw!{etd_col}2:{etd_col}{data_rows})', '"$"#,##0.00,,"M"'),
        ('Maestro Supported Revenue (ETD)', f'=SUMIF(raw!{maestro_col}2:{maestro_col}{data_rows},TRUE,raw!{etd_col}2:{etd_col}{data_rows})', '"$"#,##0.00,,"M"'),
        ('Non-Maestro Revenue (ETD)', f'=SUMIF(raw!{maestro_col}2:{maestro_col}{data_rows},FALSE,raw!{etd_col}2:{etd_col}{data_rows})', '"$"#,##0.00,,"M"'),
        ('% Revenue Supported by Maestro', f'=B7/B6', '0.0%'),
    ]
    
    for i, (label, formula, num_format) in enumerate(metrics, start=2):
        ws_metrics.cell(row=i, column=1, value=label)
        if formula:
            ws_metrics.cell(row=i, column=2, value=formula)
        if num_format:
            ws_metrics.cell(row=i, column=2).number_format = num_format
        if label:  # Bold non-empty labels
            ws_metrics.cell(row=i, column=1).font = metric_font
    
    # Create Business Rules sheet
    ws_rules = wb.create_sheet('Business Rules', 1)  # Insert after Key Metrics
    
    # Set column widths
    ws_rules.column_dimensions['A'].width = 8
    ws_rules.column_dimensions['B'].width = 80
    
    # Header row
    ws_rules['A1'] = 'Rule #'
    ws_rules['B1'] = 'Business Rule'
    ws_rules['A1'].fill = header_fill
    ws_rules['B1'].fill = header_fill
    ws_rules['A1'].font = header_font
    ws_rules['B1'].font = header_font
    
    # Section header style
    section_fill = PatternFill(start_color='D9E2F3', end_color='D9E2F3', fill_type='solid')
    section_font = Font(bold=True)
    
    # Business rules content
    rules_content = [
        (None, 'Private Client Services (PCS) Maestro Analysis - Filter Criteria', 'section'),
        ('1', 'Filter Engagement Service Code to include only 11420', 'rule'),
        ('2', 'Filter Release Date after 01/01/2025', 'rule'),
        ('3', 'Exclude engagements containing: pof, perseus, pt, ITR, BTA, or Bison (case-insensitive)', 'rule'),
        ('4', 'Exclude engagements where ANSR / Tech Revenue ETD equals 0', 'rule'),
        ('5', 'Filter Engagement Status to: Closing, Completed, Pre-Closing, Released', 'rule'),
        (None, '', 'spacer'),
        (None, 'Maestro Classification', 'section'),
        (None, 'Engagements are marked "In Maestro = TRUE" if their Engagement ID appears in the consolidated Maestro engagement data (maestro_engagement_data_consolidated.xlsx)', 'note'),
        (None, '', 'spacer'),
        (None, 'Metrics Calculation', 'section'),
        (None, '% Engagements in Maestro = Maestro Engagements / Total Engagements', 'note'),
        (None, '% Revenue Supported by Maestro = Maestro Supported Revenue / Total PCS Revenue', 'note'),
    ]
    
    for i, (rule_num, text, row_type) in enumerate(rules_content, start=2):
        if row_type == 'section':
            ws_rules.cell(row=i, column=1, value='')
            ws_rules.cell(row=i, column=2, value=text)
            ws_rules.cell(row=i, column=1).fill = section_fill
            ws_rules.cell(row=i, column=2).fill = section_fill
            ws_rules.cell(row=i, column=2).font = section_font
        elif row_type == 'rule':
            ws_rules.cell(row=i, column=1, value=rule_num)
            ws_rules.cell(row=i, column=2, value=text)
            ws_rules.cell(row=i, column=1).alignment = Alignment(horizontal='center')
        elif row_type == 'note':
            ws_rules.cell(row=i, column=1, value='')
            ws_rules.cell(row=i, column=2, value=text)
        # spacer rows are left empty

print(f'Saved to: {output_path}')
print(f'Raw data rows: {len(export_df)}')
print('Key Metrics sheet created with Excel formulas')
print('Business Rules sheet created')

Saved to: ./review/maestro/Maestro_Adoption_20260130_172129.xlsx
Raw data rows: 2909
Key Metrics sheet created with Excel formulas
Business Rules sheet created
